In [9]:
import mediapipe as mp
import cv2

In [10]:
mp_face=mp.solutions.face_mesh
mp_pose=mp.solutions.pose
mp_draw=mp.solutions.drawing_utils

In [11]:
data=mp_face.FaceMesh()
data1=mp_pose.Pose()

In [12]:
video=cv2.VideoCapture(0)
while True:
      suc,img=video.read()
      if not suc:
         break
      expression=""
      mouth_text=""
      mouth_text1=""
      face_direction=""
      head_state=""
      command=""
      img1=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
      result=data.process(img1)
      result1=data1.process(img1)
     
      if result.multi_face_landmarks:
           for i in result.multi_face_landmarks:
            
              lm=i.landmark
              upper_eye_lid_lefteye=lm[159]
              lower_eye_lid_lefteye=lm[145]

              vertical_distance_lefteye=(upper_eye_lid_lefteye.y-lower_eye_lid_lefteye.y)
            
              leftcorner_eye_lid_lefteye=lm[33]
              rightcorner_eye_lid_lefteye=lm[133]

              eye_width_lefteye=(leftcorner_eye_lid_lefteye.y)-(rightcorner_eye_lid_lefteye.y)
              eye_ratio_lefteye=vertical_distance_lefteye/eye_width_lefteye
            
              upper_eye_lid_righteye=lm[386]
              lower_eye_lid_righteye=lm[374]

              leftcorner_eye_lid_righteye=lm[362]
              rightcorner_eye_lid_righteye=lm[263]

              vertical_distance_righteye=(upper_eye_lid_righteye.y-lower_eye_lid_righteye.y)
              eye_width_righteye=(leftcorner_eye_lid_righteye.y-rightcorner_eye_lid_righteye.y)
              eye_ratio_righteye=vertical_distance_righteye/eye_width_righteye

              eye_ratio=(eye_ratio_lefteye+eye_ratio_righteye)/2

              left_eye_width  = (lm[33].x - lm[133].x)
              right_eye_width = (lm[362].x - lm[263].x)

              yaw_ratio = left_eye_width / right_eye_width
              if yaw_ratio > 1.15:
               face_direction = "ON THE LIGHTS"
              else:
               if yaw_ratio < 0.85:
                face_direction = "OPEN THE CURTAIN"

               if 0.9 < yaw_ratio < 1.1:
                if eye_ratio > 0.28:
                  expression = "EMERGENCY"
                else:
                 expression = ""

           
            
              left_mouth=lm[61]
              right_mouth=lm[291]
             
             
              upper_lip = lm[13]
              lower_lip = lm[14]
              mouth_vertical=(upper_lip.y-lower_lip.y)
              mouth_width=(left_mouth.x-right_mouth.x)

              mouth_ratio=mouth_vertical/mouth_width

              left_eye_width  = (lm[33].x - lm[133].x)
              right_eye_width = (lm[362].x - lm[263].x)



              eye_centre=(upper_eye_lid_lefteye.y+upper_eye_lid_righteye.y)/2
              nose_tip=lm[1].y
              eye_nose_difference=nose_tip-eye_centre

              if eye_nose_difference<0.02:
                 head_state="TURN ON FAN"
                 expression=""

           
            
              left_mouth=lm[61]
              right_mouth=lm[291]

              mouth_vertical=upper_lip.y-lower_lip.y
              mouth_width=left_mouth.x-right_mouth.x

              mouth_ratio=mouth_vertical/mouth_width

              if mouth_ratio > 0.1 and mouth_ratio<0.20:
                  mouth_text = "THIRSTY"
              else:
                  if mouth_ratio>0.22:
                   mouth_text="HUNGRY"
            

      if result1.pose_landmarks:
        
           lm=result1.pose_landmarks.landmark
           left_shoulder  = lm[11]
           right_shoulder = lm[12]


           diff = left_shoulder.z - right_shoulder.z

     

           if diff > 0.15:
                command+= "REMOVE CLOTHES"
           else:
                 command=""

           
           if expression:    
                cv2.putText(img,expression,(35,100),cv2.FONT_HERSHEY_DUPLEX,2,(0,0,255),2)
           elif mouth_text:
                cv2.putText(img,mouth_text,(35,100),cv2.FONT_HERSHEY_DUPLEX,2,(0,0,255),2)
           elif face_direction:
                  cv2.putText(img,face_direction,(7,100),cv2.FONT_HERSHEY_DUPLEX,2,(0,0,255),2)
           elif head_state:
                  cv2.putText(img,head_state,(35,100),cv2.FONT_HERSHEY_DUPLEX,2,(0,0,255),2)
           else:  
              cv2.putText(img,command,(34,100),cv2.FONT_HERSHEY_DUPLEX,2,(0,0,255),2)

                
               
      cv2.imshow("Face",img)
      if cv2.waitKey(1) & 0XFF==ord('q'):
          break
video.release()
cv2.destroyAllWindows()